# 05 - Search Output for Required Models

هذا هو Notebook عرض خرج المشروع المطلوب.

يختبر البحث باستخدام:
- TF-IDF
- BM25
- BERT / FAISS
- Word2Vec / FAISS
- Serial Hybrid
- Parallel Hybrid

لا نعيد كتابة الخوارزميات هنا، بل نستدعي Services الموجودة داخل `src.retrieval`.

In [ ]:
from pathlib import Path
import sys, os, json, sqlite3, subprocess, time
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# يستطيع صديقك تغيير هذا المتغير إذا كانت artifacts خارج المشروع.
# مثال في PowerShell قبل فتح notebook:
# $env:IR_ARTIFACT_ROOT="E:\\ir_project_artifacts"
ARTIFACT_ROOT = Path(os.environ.get("IR_ARTIFACT_ROOT", r"E:\ir_project_artifacts"))


def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return Path(paths[0])

DB_PATH = first_existing([
    PROJECT_ROOT / "data" / "documents.sqlite",
    PROJECT_ROOT / "data" / "documents.db",
    ARTIFACT_ROOT / "documents.sqlite",
    ARTIFACT_ROOT / "documents.db",
])

TERRIER_INDEX_PATH = first_existing([
    PROJECT_ROOT / "indexes" / "terrier_medline",
    PROJECT_ROOT / "data" / "indexes" / "terrier_medline",
    ARTIFACT_ROOT / "indexes" / "terrier_medline",
])

BERT_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_bert",
    PROJECT_ROOT / "indexes" / "faiss_bert_full",
    PROJECT_ROOT / "data" / "rag_artifacts",
    ARTIFACT_ROOT / "indexes" / "faiss_bert_full",
    ARTIFACT_ROOT / "indexes" / "faiss_bert",
])

WORD2VEC_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_word2vec",
    PROJECT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec",
])

REPORTS_DIR = PROJECT_ROOT / "reports" / "evaluation_notebook"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT       =", PROJECT_ROOT)
print("ARTIFACT_ROOT      =", ARTIFACT_ROOT)
print("DB_PATH            =", DB_PATH, "exists=", DB_PATH.exists())
print("TERRIER_INDEX_PATH =", TERRIER_INDEX_PATH, "exists=", TERRIER_INDEX_PATH.exists())
print("BERT_INDEX_DIR     =", BERT_INDEX_DIR, "exists=", BERT_INDEX_DIR.exists())
print("WORD2VEC_INDEX_DIR =", WORD2VEC_INDEX_DIR, "exists=", WORD2VEC_INDEX_DIR.exists())

In [ ]:
def file_size_mb(path):
    path = Path(path)
    if not path.exists() or path.is_dir():
        return None
    return round(path.stat().st_size / (1024 * 1024), 2)


def artifact_status_df():
    rows = []
    checks = {
        "SQLite document store": DB_PATH,
        "Terrier index folder (TF-IDF/BM25)": TERRIER_INDEX_PATH,
        "BERT index folder": BERT_INDEX_DIR,
        "BERT FAISS index": BERT_INDEX_DIR / "index.faiss",
        "BERT doc_ids.pkl": BERT_INDEX_DIR / "doc_ids.pkl",
        "BERT doc_ids.jsonl": BERT_INDEX_DIR / "doc_ids.jsonl",
        "BERT metadata.json": BERT_INDEX_DIR / "metadata.json",
        "Word2Vec index folder": WORD2VEC_INDEX_DIR,
        "Word2Vec FAISS index": WORD2VEC_INDEX_DIR / "index.faiss",
        "Word2Vec model": WORD2VEC_INDEX_DIR / "word2vec.model",
        "Word2Vec doc_ids.pkl": WORD2VEC_INDEX_DIR / "doc_ids.pkl",
        "Word2Vec metadata.json": WORD2VEC_INDEX_DIR / "metadata.json",
    }
    for name, path in checks.items():
        rows.append({"artifact": name, "path": str(path), "exists": Path(path).exists(), "size_MB": file_size_mb(path)})
    return pd.DataFrame(rows)

artifact_status_df()

In [ ]:
QUERY = "What genes are associated with breast cancer?"
TOP_K = 10

print("Query:", QUERY)
print("Top K:", TOP_K)

In [ ]:
def output_to_dataframe(output):
    rows = []
    for item in output.get("results", []):
        rows.append({
            "rank": item.get("rank"),
            "doc_id": item.get("doc_id"),
            "score": item.get("score"),
            "title": item.get("title"),
            "abstract": (item.get("abstract") or "")[:250],
        })
    return pd.DataFrame(rows)


def run_and_show(model_name, retriever):
    print("="*100)
    print("Running model:", model_name)
    print("="*100)
    try:
        output = retriever.search(QUERY)
        print("Model:", output.get("model"))
        print("Time seconds:", output.get("time_seconds"))
        df = output_to_dataframe(output)
        display(df)
        return output
    except Exception as exc:
        print("FAILED:", type(exc).__name__, exc)
        return None

In [ ]:
outputs = {}

# TF-IDF and BM25 use Terrier index.
if TERRIER_INDEX_PATH.exists() and DB_PATH.exists():
    from src.retrieval import TfidfRetriever, BM25Retriever
    outputs["tfidf"] = run_and_show("TF-IDF", TfidfRetriever(index_path=str(TERRIER_INDEX_PATH), db_path=str(DB_PATH), top_k=TOP_K))
    outputs["bm25"] = run_and_show("BM25", BM25Retriever(index_path=str(TERRIER_INDEX_PATH), db_path=str(DB_PATH), top_k=TOP_K))
else:
    print("Terrier index or SQLite is missing, cannot run TF-IDF/BM25.")
    print("Build: python scripts/build_terrier_index.py --db-path data/documents.sqlite --index-dir indexes/terrier_medline")

In [ ]:
# BERT FAISS dense retrieval
if (BERT_INDEX_DIR / "index.faiss").exists() and DB_PATH.exists():
    from src.retrieval import BertRetriever
    outputs["bert"] = run_and_show(
        "BERT/FAISS",
        BertRetriever(
            index_dir=str(BERT_INDEX_DIR),
            db_path=str(DB_PATH),
            model_name="sentence-transformers/all-MiniLM-L6-v2",
            top_k=TOP_K,
        ),
    )
else:
    print("BERT FAISS artifacts missing.")

In [ ]:
# Word2Vec FAISS retrieval
if (WORD2VEC_INDEX_DIR / "index.faiss").exists() and (WORD2VEC_INDEX_DIR / "word2vec.model").exists() and DB_PATH.exists():
    from src.retrieval import Word2VecRetriever
    outputs["word2vec"] = run_and_show(
        "Word2Vec/FAISS",
        Word2VecRetriever(index_dir=str(WORD2VEC_INDEX_DIR), db_path=str(DB_PATH), top_k=TOP_K),
    )
else:
    print("Word2Vec artifacts missing.")

In [ ]:
# Serial Hybrid: BM25 -> BERT reranking
if TERRIER_INDEX_PATH.exists() and DB_PATH.exists():
    from src.retrieval import SerialHybridRetriever
    try:
        outputs["serial_bm25_bert"] = run_and_show(
            "Serial Hybrid: BM25 -> BERT",
            SerialHybridRetriever(
                first_stage="bm25",
                second_stage="bert",
                index_path=str(TERRIER_INDEX_PATH),
                db_path=str(DB_PATH),
                word2vec_index_dir=str(WORD2VEC_INDEX_DIR),
                candidate_k=100,
                top_k=TOP_K,
            ),
        )
    except Exception as exc:
        print("Serial Hybrid failed:", type(exc).__name__, exc)
else:
    print("Terrier index missing; cannot run serial hybrid.")

In [ ]:
# Parallel Hybrid: combine available models using RRF.
available_parallel_models = []
if TERRIER_INDEX_PATH.exists():
    available_parallel_models.extend(["bm25", "tfidf"])
if (WORD2VEC_INDEX_DIR / "index.faiss").exists() and (WORD2VEC_INDEX_DIR / "word2vec.model").exists():
    available_parallel_models.append("word2vec")
if (BERT_INDEX_DIR / "index.faiss").exists():
    available_parallel_models.append("bert")

print("Available models for parallel hybrid:", available_parallel_models)

if len(available_parallel_models) >= 2 and DB_PATH.exists():
    from src.retrieval import ParallelHybridRetriever
    outputs["parallel_hybrid"] = run_and_show(
        "Parallel Hybrid / RRF",
        ParallelHybridRetriever(
            models=available_parallel_models,
            index_path=str(TERRIER_INDEX_PATH),
            db_path=str(DB_PATH),
            word2vec_index_dir=str(WORD2VEC_INDEX_DIR),
            bert_index_dir=str(BERT_INDEX_DIR),
            per_model_k=50,
            top_k=TOP_K,
            rrf_k=60,
        ),
    )
else:
    print("Not enough artifacts to run parallel hybrid.")